# Exploring the spatial atlas

Interactive tour of the Visium side of the project: **69,118 spots over 27 sections from 4 cohorts**, each spot
deconvolved with cell2location against a 13-class reference built from the single-cell atlas
(5 malignant metaprogram families + hypoxia + novel_MP9 + 5 microenvironment classes).

There is **no single merged spatial object** — spatial data lives per section:

| product | path | note |
|---|---|---|
| per-section abundances (authoritative) | `data/processed/spatial/<cohort>/<section>_abundance.csv` | 27 files, q05 + posterior mean for 13 classes |
| collected long form | `results/tables/spatial_abundance_long.csv` | 898,534 rows = 69,118 spots x 13 classes (126 MB) |
| niche labels | `data/processed/spatial/spatial_niche_labels.csv.gz` | 52,774 spots -> niche, patient_id; **DMG only** |
| section inventory | `results/qc/spatial_sections_used.csv` | spots, genes, scale factors per section |
| Visium inputs (pre-deconvolution) | `modal/inputs/visium_<cohort>.h5ad` | staged for the GPU job |

> **Do not use `data/processed/spatial/spatial_deconvolution_*.h5ad`.** Those four files are from the 17 Oct run and
> carry only **11** classes — they predate the audit fix that added `malignant_hypoxia` and `malignant_novel_MP9` and
> re-ran every cohort to 20,000 epochs. This notebook reads the 20 Oct per-section CSVs instead.

> **Niches and `patient_id` cover the DMG spots only.** The niche-label file holds 52,774 of the 69,118 spots; the
> 16,344 missing ones are exactly the six glioblastoma sections of the Ren cohort (GBM1-4, GBM5_1, GBM5_2), which
> were never assigned a DMG patient or a niche. Any grouping by `patient_id` or `niche` therefore describes
> 12 DMG patients across 21 sections and silently excludes glioblastoma - Section 7 uses the section labels instead.

Runs in a few seconds and needs < 1 GB of RAM — no GPU, no h5ad loading.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, fixed

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / "data" / "processed").exists())
SPATIAL, TABLES = ROOT / "data/processed/spatial", ROOT / "results/tables"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

CLASSES = ["malignant_AC_like", "malignant_OPC_like", "malignant_OC_like", "malignant_MES_like",
           "malignant_cycling", "malignant_hypoxia", "malignant_novel_MP9",
           "astrocyte", "oligodendrocyte", "neuron", "myeloid", "T_NK", "vascular"]
MALIGNANT = [c for c in CLASSES if c.startswith("malignant_")]
print("repo:", ROOT)

## 1. Assemble the spot table

The 27 per-section CSVs are concatenated and joined to the niche labels (which carry `patient_id`) and to the
section inventory (which carries the region or diagnosis). Nothing is recomputed — these are the posterior
summaries written by the cell2location run.

In [ ]:
files = sorted(SPATIAL.glob("*/*_abundance.csv"))
spots = pd.concat([pd.read_csv(f, index_col=0) for f in files])
spots.index.name = "spot"
niche = pd.read_csv(SPATIAL / "spatial_niche_labels.csv.gz").set_index("spot")
spots = spots.join(niche[["patient_id", "niche"]])
inv = pd.read_csv(ROOT / "results/qc/spatial_sections_used.csv")
spots = spots.join(inv.set_index("section_id")["region_or_diagnosis"], on="section_id")

Q = spots[[f"q05_{c}" for c in CLASSES]].rename(columns=lambda c: c[4:])
M = spots[[f"mean_{c}" for c in CLASSES]].rename(columns=lambda c: c[5:])
PROP = M.div(M.sum(1), axis=0)                      # composition, the primary metric of the paper
spots["tumour_fraction"] = PROP[MALIGNANT].sum(1)

print(f"{len(files)} section files -> {len(spots):,} spots, {spots.section_id.nunique()} sections, "
      f"{spots.cohort.nunique()} cohorts")
unl = spots.patient_id.isna()
print(f"niche / patient_id assigned to {(~unl).sum():,} spots ({spots.patient_id.nunique()} DMG patients, "
      f"{spots.section_id[~unl].nunique()} sections); {unl.sum():,} unlabelled spots in "
      f"{sorted(spots.section_id[unl].unique())}")
print("classes:", len(CLASSES), "| columns:", [c for c in spots.columns if not c.startswith(("q05_", "mean_"))])
spots.head(3)[["section_id", "cohort", "patient_id", "niche", "n_counts", "n_genes", "tumour_fraction"]]

In [ ]:
summary = (spots.groupby(["cohort", "section_id"], observed=True)
           .agg(patient=("patient_id", "first"), region=("region_or_diagnosis", "first"),
                spots=("n_counts", "size"), median_counts=("n_counts", "median"),
                median_genes=("n_genes", "median"), tumour_fraction=("tumour_fraction", "median"))
           .round(3))
display(summary)
print("sections per cohort:", spots.groupby("cohort", observed=True).section_id.nunique().to_dict())

## 2. One class on one section

Spots are plotted on Visium pixel coordinates (row inverted to match the slide orientation). The histology images
are not bundled with these tables, so this is the bare spot grid. `q05` is the 5th-percentile posterior abundance
cell2location reports — a conservative estimate; `mean` is the posterior mean; `proportion` is the mean normalised
per spot, which is what every statistic in the manuscript uses.

In [ ]:
def plot_section(section, cls="malignant_AC_like", metric="proportion", ax=None, s=None, cmap="magma_r"):
    d = spots[spots.section_id == section]
    v = {"q05": Q, "mean": M, "proportion": PROP}[metric].loc[d.index, cls]
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4.6))
    if s is None:
        s = max(1.0, 9000 / len(d))
    h = ax.scatter(d.pxl_col, -d.pxl_row, c=v, s=s, cmap=cmap, lw=0, rasterized=True,
                   vmin=0, vmax=max(np.quantile(v, 0.99), 1e-9))
    plt.colorbar(h, ax=ax, shrink=0.65, label=metric)
    ax.set(xticks=[], yticks=[], aspect="equal",
           title=f"{section} - {cls}\n{d.region_or_diagnosis.iloc[0][:48]}")
    for sp in ax.spines.values():
        sp.set_visible(False)
    return ax

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for cls, ax in zip(["malignant_AC_like", "myeloid", "oligodendrocyte"], axes):
    plot_section("Pons_A", cls, ax=ax)
fig.tight_layout()

In [ ]:
@interact(section=Dropdown(options=sorted(spots.section_id.unique()), value="DMG1", description="section"),
          cls=Dropdown(options=CLASSES, value="malignant_AC_like", description="class"),
          metric=Dropdown(options=["proportion", "mean", "q05"], value="proportion", description="metric"))
def _one(section, cls, metric):
    plot_section(section, cls, metric)
    plt.show()

## 3. All thirteen classes on one section

In [ ]:
def section_panel(section, metric="proportion", ncol=5):
    d = spots[spots.section_id == section]
    nrow = int(np.ceil(len(CLASSES) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.5 * nrow))
    for ax, cls in zip(axes.ravel(), CLASSES):
        v = {"q05": Q, "mean": M, "proportion": PROP}[metric].loc[d.index, cls]
        ax.scatter(d.pxl_col, -d.pxl_row, c=v, s=max(0.6, 4000 / len(d)), cmap="magma_r", lw=0,
                   rasterized=True, vmin=0, vmax=max(np.quantile(v, 0.99), 1e-9))
        ax.set(xticks=[], yticks=[], aspect="equal", title=cls.replace("malignant_", ""), frame_on=False)
        ax.title.set_fontsize(8)
    for ax in axes.ravel()[len(CLASSES):]:
        ax.axis("off")
    fig.suptitle(f"{section} - {d.region_or_diagnosis.iloc[0]} ({len(d):,} spots, {metric})", y=1.0, fontsize=10)
    fig.tight_layout()
    return fig

section_panel("DMG1");

In [ ]:
@interact(section=Dropdown(options=sorted(spots.section_id.unique()), value="Pons_A"),
          metric=Dropdown(options=["proportion", "mean", "q05"], value="proportion"))
def _panel(section, metric):
    section_panel(section, metric)
    plt.show()

## 4. Composition across sections

Mean per-spot composition per section. Ren contributes the only out-group in the project: 5 glioblastoma sections
alongside 5 DMG sections and 1 peritumour section.

In [ ]:
comp = PROP.join(spots[["cohort", "section_id"]]).groupby(["cohort", "section_id"], observed=True).mean()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.6, 1]})
comp.plot.bar(stacked=True, ax=axes[0], width=0.85, colormap="tab20", legend=False)
axes[0].set(ylabel="mean spot composition", xlabel="", ylim=(0, 1))
axes[0].legend(frameon=False, fontsize=6, loc="center left", bbox_to_anchor=(1.0, 0.5))
axes[0].set_xticklabels([t.get_text().split(", ")[1].rstrip(")") for t in axes[0].get_xticklabels()],
                        rotation=90, fontsize=7)

tf = spots.groupby("section_id", observed=True).tumour_fraction.median().sort_values()
tf.plot.barh(ax=axes[1], color=["#de2d26" if s.startswith("DMG") else "#4a5568" for s in tf.index])
axes[1].set(xlabel="median tumour fraction per spot", ylabel="")
axes[1].tick_params(labelsize=7)
fig.tight_layout()

## 5. Co-localisation

Per-section Spearman correlations between class proportions, pooled across **12 patients and 21 sections** with a
random-effects meta-analysis, Benjamini-Hochberg over the 42 malignant-state x microenvironment pairs. Two
hypotheses were stated before looking: MES-like with myeloid **holds** (rho 0.238, q 0.034) and OPC-like with
oligodendrocyte **fails** (rho 0.178, q 0.210). Between-section heterogeneity is extreme throughout
(I^2 0.952-0.997), so the sign, not the magnitude, is the result.

In [ ]:
co_ = pd.read_csv(TABLES / "spatial_colocalisation.csv")
sel = co_[(co_.metric == "proportion") & (co_.group == "DMG") & (co_.pair_type == "state_vs_TME")]
display(sel.sort_values("q_bh_state_vs_TME")
        [["class_a", "class_b", "rho_re", "rho_re_lo", "rho_re_hi", "p_re", "q_bh_state_vs_TME",
          "n_patients", "n_sections", "n_sections_positive", "I2"]]
        .head(12).round(4).set_index(["class_a", "class_b"]))

piv = sel.pivot(index="class_a", columns="class_b", values="rho_re")
qpiv = sel.pivot(index="class_a", columns="class_b", values="q_bh_state_vs_TME")
fig, ax = plt.subplots(figsize=(6.4, 5))
im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
ax.set(xticks=range(piv.shape[1]), yticks=range(piv.shape[0]), yticklabels=piv.index)
ax.set_xticklabels(piv.columns, rotation=45, ha="right")
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        rho, q = piv.values[i, j], qpiv.values[i, j]
        star = "*" if (q == q and q < 0.05) else ""
        ax.text(j, i, f"{rho:.2f}{star}", ha="center", va="center", fontsize=7,
                color="white" if abs(rho) > 0.3 else "black")
plt.colorbar(im, ax=ax, shrink=0.75, label="random-effects rho")
ax.set_title("* = BH q < 0.05 over 42 pairs", fontsize=8)
fig.tight_layout()

## 6. The brainstem gradient

The brainstem cohort is 10 serial sections through **one** DIPG autopsy — an anatomical series, not a cohort, so
every number here describes a single patient. Within sections, tumour fraction rises with AC-like (+0.285) and
OPC-like (+0.213) share and falls with astrocyte (-0.431), MES-like (-0.385) and oligodendrocyte (-0.298).

In [ ]:
grad = pd.read_csv(TABLES / "brainstem_gradient.csv")
re_ = grad[grad.test.str.endswith("_RE")][["test", "cls", "rho_re", "rho_re_lo", "rho_re_hi", "p_re",
                                           "n_sections", "n_sections_positive", "q_bh_re"]]
display(re_.sort_values("rho_re").round(4).set_index(["test", "cls"]))

bs = spots[spots.cohort == "brainstem"]
order = bs.groupby("section_id", observed=True).tumour_fraction.median().sort_values(ascending=False).index
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].boxplot([bs.tumour_fraction[bs.section_id == s] for s in order], tick_labels=order, showfliers=False)
axes[0].set(ylabel="tumour fraction per spot", title="brainstem sections, ranked")
axes[0].tick_params(axis="x", rotation=60, labelsize=7)
for cls, col in zip(["malignant_AC_like", "malignant_OPC_like", "astrocyte", "oligodendrocyte"],
                    ["#de2d26", "#fd8d3c", "#2b6cb0", "#38a169"]):
    axes[1].scatter(bs.tumour_fraction, PROP.loc[bs.index, cls], s=1, alpha=0.15, color=col, rasterized=True,
                    label=cls)
axes[1].set(xlabel="tumour fraction", ylabel="class proportion", title="within-section relationships")
axes[1].legend(frameon=False, fontsize=7, markerscale=6)
fig.tight_layout()

## 7. DMG against glioblastoma (Ren cohort)

The only cross-diagnosis contrast available: 5 DMG against 5 glioblastoma sections. Neither difference survives
correction — tumour fraction 0.528 vs 0.460 (p = 0.056, q = 0.667) and cycling 0.112 vs 0.214 (p = 0.095,
q = 0.667). With n = 5 per arm this is a descriptive comparison.

In [ ]:
ren = pd.read_csv(TABLES / "ren_dmg_vs_gbm_composition.csv")
display(ren[["variable", "dmg_median", "gbm_median", "n_dmg", "n_gbm", "p_mwu", "q_bh"]].round(4)
        .set_index("variable"))

r = spots[spots.cohort == "ren"].copy()
r["arm"] = np.where(r.section_id.str.startswith("DMG"), "DMG",
                    np.where(r.region_or_diagnosis.str.contains("peritumour", case=False), "peritumour", "GBM"))
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for i, arm in enumerate(["DMG", "GBM", "peritumour"]):
    d = r[r.arm == arm]
    ax.scatter(np.full(d.section_id.nunique(), i) + np.linspace(-0.12, 0.12, d.section_id.nunique()),
               d.groupby("section_id", observed=True).tumour_fraction.median(), s=40, label=arm)
ax.set(xticks=[0, 1, 2], xticklabels=["DMG (5)", "GBM (5)", "peritumour (1)"],
       ylabel="median tumour fraction", title="section-level medians")
fig.tight_layout()

## 8. Niches

Spots were clustered on their composition into shared niches, then compared across cohorts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 1.1]})
nm = PROP.join(spots["niche"]).groupby("niche", observed=True).mean()
im = axes[0].imshow(nm.values, cmap="viridis", aspect="auto")
axes[0].set(xticks=range(nm.shape[1]), yticks=range(nm.shape[0]), yticklabels=nm.index, ylabel="niche",
            title="mean composition per niche")
axes[0].set_xticklabels(nm.columns, rotation=45, ha="right", fontsize=7)
plt.colorbar(im, ax=axes[0], shrink=0.7, label="proportion")

d = spots[spots.section_id == "Pons_A"]
for n, g in d.groupby("niche", observed=True):
    axes[1].scatter(g.pxl_col, -g.pxl_row, s=2, label=f"niche {n} ({len(g):,})", lw=0, rasterized=True)
axes[1].set(xticks=[], yticks=[], aspect="equal", title="Pons_A niches")
axes[1].legend(frameon=False, fontsize=7, markerscale=5, loc="center left", bbox_to_anchor=(1, 0.5))
for sp in axes[1].spines.values():
    sp.set_visible(False)
fig.tight_layout()

@interact(section=Dropdown(options=sorted(spots.section_id.unique()), value="DMG1"))
def _niche(section):
    d = spots[spots.section_id == section]
    fig, ax = plt.subplots(figsize=(5, 4.6))
    for n, g in d.groupby("niche", observed=True):
        ax.scatter(g.pxl_col, -g.pxl_row, s=max(1, 6000 / len(d)), label=f"{n} ({len(g):,})", lw=0, rasterized=True)
    ax.set(xticks=[], yticks=[], aspect="equal", title=section)
    ax.legend(frameon=False, fontsize=7, markerscale=4, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.show()

## 9. Did the deconvolution converge?

Recomputed here from the four harvested ELBO traces rather than read from
`results/tables/c2l_elbo_plateau.csv` — that stored table holds only three cohorts, because `ren` was harvested
after it was written (its trace is present in `results/tables/c2l_histories/`). Ren is also the one cohort that
stopped short of the 20,000-epoch target, at 12,940 epochs, when it hit the 2,400 s wall-clock budget in
`modal/phase6_c2l.py`. Its trace is nonetheless flat: the last 5 % of logged epochs account for 0.022 % of the
total ELBO reduction.

In [ ]:
rows = []
for f in sorted((TABLES / "c2l_histories").glob("*_history.csv")):
    if f.stem.startswith("signatures"):
        continue
    h = pd.read_csv(f)
    e = h["elbo_train"].values.astype(float)
    rows.append({"cohort": f.stem.replace("_history", ""), "epochs": len(e), "elbo_first": e[0],
                 "elbo_last": e[-1], "pct_reduction_in_last_5pct": 100 * (e[int(len(e) * .95)] - e[-1]) / (e[0] - e[-1]),
                 "in_stored_table": f.stem.replace("_history", "") in
                                    set(pd.read_csv(TABLES / "c2l_elbo_plateau.csv").cohort)})
display(pd.DataFrame(rows).set_index("cohort").round(4))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for f in sorted((TABLES / "c2l_histories").glob("*_history.csv")):
    if f.stem.startswith("signatures"):
        continue
    h = pd.read_csv(f)
    ax.plot(h.epoch, h.elbo_train / h.elbo_train.iloc[0], lw=1, label=f.stem.replace("_history", ""))
ax.set(xlabel="epoch", ylabel="ELBO / initial ELBO", title="training traces (normalised)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## Caveats

- **Proportions are the primary metric.** Absolute cell2location abundances scale with the `N_cells_per_location`
  prior (5) and with per-spot sequencing depth, so comparisons across cohorts use composition.
- **Patients, not sections, are the unit.** The co-localisation and gradient statistics pool sections within patient
  with random effects; `n_sections_positive` reports how many sections agree in sign.
- **One brainstem patient.** The rostro-caudal series is a single autopsy — anatomy, not epidemiology.
- **The reference is this atlas.** Spatial classes are defined by the single-cell metaprogram families, so spatial
  and single-cell results are not independent evidence of the same thing.
- **No histology images** accompany the abundance tables; `modal/inputs/visium_<cohort>.h5ad` holds the original
  Visium objects with their scale factors if you need to overlay tissue.

Companion notebook: `notebooks/single_cell_atlas.ipynb`. Statistics and methods: `docs/phase6_notes.md`.